In [1]:
import random
import numpy as np
from tqdm import tqdm

In [42]:
class Maze:
    def __init__(self):
        self.maze = [
            "OOOOOOOCG",
            "OOCOOOOCO",
            "SOCOOOOCO",
            "OOCOOOOOO",
            "OOOOOCOOO",
            "OOOOOOOOO",
        ]
        self.starting_state = (2,0)
        self.goal_state = (0, 8)

        self.number_of_steps = 0
        self.current_state = self.starting_state

        self.terminated = 0
        self.truncated = 0

    def get_action_space(self):
        return ["up", "down", "left", "right"]

    def print_maze(self):
        print(self.maze)

    def change_state(self, action):

        new_state = (0, 0)

        if action == "up":
            if self.current_state[0] == 0:
                return
            new_state = (self.current_state[0] - 1, self.current_state[1])
        elif action == "down":
            if self.current_state[0] == 5:
                return
            new_state = (self.current_state[0] + 1, self.current_state[1])
        elif action == "left":
            if self.current_state[1] == 0:
                return
            new_state = (self.current_state[0], self.current_state[1] - 1)
        elif action == "right":
            if self.current_state[1] == 8:
                return

            new_state = (self.current_state[0], self.current_state[1] + 1)

        if (
            self.maze[new_state[0]][new_state[1]] == "O"
            or self.maze[new_state[0]][new_state[1]] == "S"
        ):
            self.current_state = new_state
        elif self.maze[new_state[0]][new_state[1]] == "G":
            self.current_state = new_state
            # print("Goal reached!")

    def step(self, action):

        if self.terminated or self.truncated:
            observation = self.current_state
            reward = 0
            return observation, reward, self.terminated, self.truncated

        self.number_of_steps += 1

        self.change_state(action)

        observation = self.current_state

        reward = 0
        if self.maze[self.current_state[0]][self.current_state[1]] == "G":
            reward = 1

            self.terminated  = 1
        else:
            reward = 0

        if self.number_of_steps == 100000:
            self.truncated = 1

        return observation, reward, self.terminated, self.truncated

    def reset(self):
        self.number_of_steps = 0
        self.current_state = self.starting_state

        self.terminated = 0
        self.truncated = 0

In [43]:
class Dyna_Q:
    def __init__(self, env, n, number_of_episodes = 100):
        self.states_and_actions_taken = {}
        self.env = env
        self.n = n

        self.Q = {}
        self.Model = {}

        self.initialize_Q_and_Model()

        self.number_of_episodes = number_of_episodes

    def initialize_Q_and_Model(self):

        for row in range(6):
            for col in range(9):
                self.Q[(row, col)] = {"up": 0, "down": 0, "left": 0, "right": 0}
                self.Model[(row, col)] = {"up": (0,0), "down": (0,0), "left": (0,0), "right": (0,0)}


    def get_argmax(self, S):
        state_dict = self.Q[S]

        max_value = max(state_dict.values())
        return [action for action, value in state_dict.items() if value == max_value]



    def get_max(self, S):
        state_dict = self.Q[S]

        action = max(state_dict, key=state_dict.get)
        return self.Q[S][action]


    def get_action(self, epsilon, S):
        action = random.choice(self.env.get_action_space())
        if random.random() > epsilon:
            possible_actions = self.get_argmax(S)
            action = random.choice(possible_actions)

        return action

    def update_state_action_collection(self, S, A):
        if S not in self.states_and_actions_taken:
            self.states_and_actions_taken[S] = []

        if A not in self.states_and_actions_taken[S]:
            self.states_and_actions_taken[S].append(A)

    def get_policy(self):
        policy = {}
        for S in self.Q_Table:
            A = self.get_argmax(self.Q_Table[S])

            policy[S] = A
        return policy




    def do_planning_updates(self):
        for _ in range(self.n):

            S = random.choice(list(self.states_and_actions_taken.keys()))
            A = random.choice(self.states_and_actions_taken[S])


            R, S_prime = self.Model[S][A]

            self.Q[S][A] = self.Q[S][A] + 0.1 * (
                R + self.get_max(S_prime) - self.Q[S][A]
            )

    def do_one_dyna_Q_iteration(self):
        S = self.env.current_state
        R = 0
        S_prime = self.env.current_state
        terminated = 0
        truncated = 0
        self.env.reset()


        trajectory = []

        while not terminated and not truncated:
            S = S_prime
            trajectory.append(S)
            A = self.get_action(0.2, S)

            self.update_state_action_collection(S, A)

            S_prime, R, terminated, truncated  = self.env.step(A)




            self.Q[S][A] = self.Q[S][A] + 0.1* (R + self.get_max(S_prime) - self.Q[S][A])




            self.Model[S][A] = (R, S_prime)


            self.do_planning_updates()



    def do_dyna_Q(self):
        for i in tqdm(range(self.number_of_episodes)):
            self.do_one_dyna_Q_iteration()

In [44]:
maze = Maze()


In [7]:
dyna = Dyna_Q(maze, n=5)

In [41]:
random.choice(dyna.get_argmax((0,1)))

'left'

In [587]:
dyna.do_dyna_Q()

100%|██████████| 100/100 [00:41<00:00,  2.40it/s]


In [13]:
dyna.Q[(0,0)]["down"] = 1

In [532]:
dyna.Q[(1,0)]["right"] = 4

In [541]:
dyna.update_state_action_collection((2,0),"up")

In [542]:
dyna.states_and_actions_taken

{(3, 0): ['left', 'up', 'down'],
 (3, 1): ['down'],
 (1, 0): ['down'],
 (2, 0): ['down', 'up']}

In [554]:
dyna.Model[(2, 0)]["up"] = (0, (1, 0))

In [555]:
dyna.Model

{(0, 0): {'up': (0, 0), 'down': (0, 0), 'left': (0, 0), 'right': (0, 0)},
 (0, 1): {'up': (0, 0), 'down': (0, 0), 'left': (0, 0), 'right': (0, 0)},
 (0, 2): {'up': (0, 0), 'down': (0, 0), 'left': (0, 0), 'right': (0, 0)},
 (0, 3): {'up': (0, 0), 'down': (0, 0), 'left': (0, 0), 'right': (0, 0)},
 (0, 4): {'up': (0, 0), 'down': (0, 0), 'left': (0, 0), 'right': (0, 0)},
 (0, 5): {'up': (0, 0), 'down': (0, 0), 'left': (0, 0), 'right': (0, 0)},
 (0, 6): {'up': (0, 0), 'down': (0, 0), 'left': (0, 0), 'right': (0, 0)},
 (0, 7): {'up': (0, 0), 'down': (0, 0), 'left': (0, 0), 'right': (0, 0)},
 (0, 8): {'up': (0, 0), 'down': (0, 0), 'left': (0, 0), 'right': (0, 0)},
 (1, 0): {'up': (0, 0), 'down': (0, (2, 0)), 'left': (0, 0), 'right': (0, 0)},
 (1, 1): {'up': (0, 0), 'down': (0, 0), 'left': (0, 0), 'right': (0, 0)},
 (1, 2): {'up': (0, 0), 'down': (0, 0), 'left': (0, 0), 'right': (0, 0)},
 (1, 3): {'up': (0, 0), 'down': (0, 0), 'left': (0, 0), 'right': (0, 0)},
 (1, 4): {'up': (0, 0), 'down': (

In [556]:
dyna.do_planning_updates()

In [557]:
dyna.Q

{(0, 0): {'up': 0, 'down': 0, 'left': 0, 'right': 0},
 (0, 1): {'up': 0, 'down': 0, 'left': 0, 'right': 0},
 (0, 2): {'up': 0, 'down': 0, 'left': 0, 'right': 0},
 (0, 3): {'up': 0, 'down': 0, 'left': 0, 'right': 0},
 (0, 4): {'up': 0, 'down': 0, 'left': 0, 'right': 0},
 (0, 5): {'up': 0, 'down': 0, 'left': 0, 'right': 0},
 (0, 6): {'up': 0, 'down': 0, 'left': 0, 'right': 0},
 (0, 7): {'up': 0, 'down': 0, 'left': 0, 'right': 0},
 (0, 8): {'up': 0, 'down': 0, 'left': 0, 'right': 0},
 (1, 0): {'up': 0, 'down': 0, 'left': 0, 'right': 4},
 (1, 1): {'up': 0, 'down': 0, 'left': 0, 'right': 0},
 (1, 2): {'up': 0, 'down': 0, 'left': 0, 'right': 0},
 (1, 3): {'up': 0, 'down': 0, 'left': 0, 'right': 0},
 (1, 4): {'up': 0, 'down': 0, 'left': 0, 'right': 0},
 (1, 5): {'up': 0, 'down': 0, 'left': 0, 'right': 0},
 (1, 6): {'up': 0, 'down': 0, 'left': 0, 'right': 0},
 (1, 7): {'up': 0, 'down': 0, 'left': 0, 'right': 0},
 (1, 8): {'up': 0, 'down': 0, 'left': 0, 'right': 0},
 (2, 0): {'up': 0, 'down': 0

In [326]:
dyna.get_argmax((1,0))

'right'

In [481]:
dyna.get_action(0.2, (1,0))

'right'

In [280]:
dyna.Q[(1,0)]

{'up': 0, 'down': 0, 'left': 0, 'right': 4}